In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Load and pre-process data

In [2]:
target_col = "log_average_ms"

node_cols = ["Abs", "Acos", "Add", "ai.onnx.ml::CategoryMapper", "And", "ArgMax", 
             "AveragePool", "BatchNormalization", "Cast", "Ceil", "Clip", 
             "com.microsoft::BiasGelu", "com.microsoft::DynamicQuantizeLSTM", 
             "com.microsoft::DynamicQuantizeMatMul", "com.microsoft::FastGelu", 
             "com.microsoft::FusedConv", "com.microsoft::FusedGemm", 
             "com.microsoft::FusedMatMul", "com.microsoft::MatMulIntegerToFloat", 
             "com.microsoft::QGemm", "com.microsoft::QLinearAdd", 
             "com.microsoft::QLinearAveragePool", "com.microsoft::QLinearConcat", 
             "com.microsoft::QLinearGlobalAveragePool", "com.microsoft::QLinearLeakyRelu", 
             "com.microsoft::QLinearMul", "com.microsoft::QLinearSigmoid", 
             "com.microsoft::QuickGelu", "com.microsoft::SkipLayerNormalization", 
             "Compress", "Concat", "Constant", "ConstantOfShape", "Conv", "ConvInteger", 
             "ConvTranspose", "Cos", "CumSum", "DequantizeLinear", "Div", "Dropout", 
             "DynamicQuantizeLinear", "Einsum", "Equal", "Erf", "Exp", "Expand", 
             "EyeLike", "Flatten", "Floor", "Gather", "GatherElements", "GatherND", 
             "Gelu", "Gemm", "GlobalAveragePool", "Greater", "GreaterOrEqual", "Hardmax", 
             "HardSigmoid", "HardSwish", "Identity", "If", "InstanceNormalization", 
             "LayerNormalization", "LeakyRelu", "Less", "LessOrEqual", "local::preprocess", 
             "Log", "LogSoftmax", "Loop", "LRN", "LSTM", "MatMul", "MatMulInteger", 
             "Max", "MaxPool", "Min", "Mod", "Mul", "Neg", "NonMaxSuppression", "NonZero", 
             "Not", "OneHot", "Or", "Pad", "Pow", "PRelu", "QLinearConv", "QLinearMatMul", 
             "QuantizeLinear", "Range", "Reciprocal", "ReduceMax", "ReduceMean", "ReduceMin", 
             "ReduceProd", "ReduceSum", "Relu", "Reshape", "Resize", "RoiAlign", "Round", 
             "Scan", "ScatterElements", "ScatterND", "Shape", "Sigmoid", 
             "SimplifiedLayerNormalization", "Sin", "Slice", "Softmax", "Split", "Sqrt", 
             "Squeeze", "Sub", "Sum", "Tanh", "Tile", "TopK", "Transpose", "Trilu", 
             "Unsqueeze", "Where", "Xor"]

feature_cols = node_cols + ["conv_flops", "matmul_flops",
                "elementwise_mb", "reduction_mb", "normalization_mb",
                "movement_mb", "memory_mb", "l1d_cache_kb",
                "l1i_cache_kb", "l2_cache_kb", "base_clock_mhz",
                "num_cores", "memory_bandwith_gbs"]

metadata_cols = ["model", "cpu_provider",
                 "machine_type", "platform", "run_id"]

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# load test, train, val sets

train_df = pd.read_csv("/content/drive/MyDrive/Data/train_set.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Data/val_set.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Data/test_set.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (100252, 154)
val: (21483, 154)
test: (21483, 154)


In [6]:
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns)

train_df.head()

,model,input_dimensions,input_dtypes,output_dimensions,output_dtypes,conv_flops,matmul_flops,elementwise_mb,reduction_mb,normalization_mb,...,base_clock_mhz,memory_bandwith_gbs,cpu_provider,machine_type,platform,repo_file,average_ms,stddev_ms,min_ms,max_ms
0,hardcorenas_d_Opset17_extended.onnx,x:1x3x224x224,x:float32,668:1x1000,668:float32,472440512,2560000,50.528229,5.204269,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,hardcorenas_d_Opset17.onnx,14.069188,0.082284,13.931521,14.247209
1,vit_base_patch8_224_in21k_Opset17_disable_all....,x:1x3x224x224,x:float32,1089:1x21843,1089:float32,231211008,156097479168,1771.169711,0.000000,792.141724,...,2450.000,205,amd,epyc,gcloud,vit_base_patch8_224_in21k_Opset17.onnx,1481.456045,150.199862,1094.324125,1632.506909
2,tf_efficientnetv2_m_in21ft1k_Opset17_basic.onnx,x:1x3x384x384,x:float32,2466:1x1000,2466:float32,31470991360,2560000,1201.339462,67.736023,0.000000,...,2449.998,205,amd,epyc,gcloud,tf_efficientnetv2_m_in21ft1k_Opset17.onnx,301.127424,12.889699,286.464729,324.336829
3,shufflenet_v2_x1_5_Opset16.onnx,x:1x3x224x224,x:float32,1137:1x1000,1137:float32,266034528,2048000,8.246674,1.630875,0.000000,...,2449.998,205,amd,epyc,gcloud,shufflenet_v2_x1_5_Opset16.onnx,8.161246,0.036559,8.080030,8.197800
4,wide_resnet101_2_Opset16_timm.onnx,x:1x3x224x224,x:float32,971:1x1000,971:float32,45502005248,4096000,252.656250,4.218750,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,wide_resnet101_2_Opset16_timm.onnx,38.147223,0.068437,38.035275,38.292409


In [7]:
# pre-processing function

def preprocess(df: pd.DataFrame, train=False) -> pd.DataFrame:

  #drop rows with missing features
  df = df.dropna(subset=["average_ms"])

  #remove rows with high standard deviation (only for train)
  cv_limit = 0.2 # 20% cv limit
  df["cv"] = df["stddev_ms"] / df["average_ms"]

  if train:
    df = df[df["cv"] <= cv_limit].copy()

  #add log_average_ms column
  df["log_average_ms"] = np.log(df["average_ms"])

  #remove decimals and round down
  for col in ["elementwise_mb", "reduction_mb", "normalization_mb", "movement_mb"]:
    df[col] = df[col].round(0)

  return df

In [8]:
train_df = preprocess(train_df, True)
test_df = preprocess(test_df, False)
val_df = preprocess(val_df, False)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (98628, 156)
val: (21483, 156)
test: (21483, 156)


# Model Training

In [9]:
from sklearn.model_selection import ParameterSampler

In [10]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [11]:
def rmse_log(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

In [12]:
def train_rf(params, X_train, y_train, X_val, y_val):
    model = RandomForestRegressor(
        random_state=67,
        n_jobs=-1,
        **params,
    )

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    score = rmse_log(y_val, val_pred)

    return model, score

In [13]:
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [10, 15, 20, 30, 40, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5, 0.7, 0.9, 1.0],
    "bootstrap": [True],
}

In [14]:
sampled_params = list(ParameterSampler(
    param_dist,
    n_iter=30,
    random_state=67,
))

In [15]:
results = []

best_model = None
best_score = float("inf")
best_params = None

for i, params in enumerate(sampled_params, start=1):
    model, val_rmse = train_rf(params, X_train, y_train, X_val, y_val)

    row = {
        "run": i,
        "val_rmse_log": val_rmse,
        **params,
    }

    results.append(row)

    if val_rmse < best_score:
        best_score = val_rmse
        best_model = model
        best_params = params

    print(f"{i:02d} | val RMSE log = {val_rmse:.5f} | best = {best_score:.5f}")

01 | val RMSE log = 0.38074 | best = 0.38074
02 | val RMSE log = 0.22394 | best = 0.22394
03 | val RMSE log = 0.22295 | best = 0.22295
04 | val RMSE log = 0.25826 | best = 0.22295
05 | val RMSE log = 0.37959 | best = 0.22295
06 | val RMSE log = 0.40580 | best = 0.22295
07 | val RMSE log = 0.31113 | best = 0.22295
08 | val RMSE log = 0.42938 | best = 0.22295
09 | val RMSE log = 0.22517 | best = 0.22295
10 | val RMSE log = 0.22786 | best = 0.22295
11 | val RMSE log = 0.21724 | best = 0.21724
12 | val RMSE log = 0.24904 | best = 0.21724
13 | val RMSE log = 0.24363 | best = 0.21724
14 | val RMSE log = 0.39732 | best = 0.21724
15 | val RMSE log = 0.25767 | best = 0.21724
16 | val RMSE log = 0.29755 | best = 0.21724
17 | val RMSE log = 0.23988 | best = 0.21724
18 | val RMSE log = 0.31596 | best = 0.21724
19 | val RMSE log = 0.22051 | best = 0.21724
20 | val RMSE log = 0.67801 | best = 0.21724
21 | val RMSE log = 0.46834 | best = 0.21724
22 | val RMSE log = 0.21992 | best = 0.21724
23 | val R

In [16]:
tuning_results = pd.DataFrame(results).sort_values("val_rmse_log")
tuning_results.head(10)

,run,val_rmse_log,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap
10,11,0.217240,800,2,1,0.5,NaN,True
28,29,0.218700,100,10,1,0.7,40.0,True
21,22,0.219921,800,10,1,0.9,30.0,True
18,19,0.220510,800,5,2,1.0,30.0,True
2,3,0.222947,100,10,2,1.0,20.0,True
1,2,0.223940,500,5,5,0.5,NaN,True
8,9,0.225172,100,20,1,0.5,20.0,True
29,30,0.225998,500,20,1,0.9,NaN,True
9,10,0.227861,300,20,5,0.7,20.0,True
16,17,0.239881,500,2,10,0.9,20.0,True


# Evaluation

In [17]:
final_model = best_model

test_pred = final_model.predict(X_test)
test_rmse_log = np.sqrt(mean_squared_error(y_test, test_pred))

print("Test RMSE: ", test_rmse_log)

Test RMSE:  0.21057408955092471


In [18]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,

        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),

        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,

        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),

        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

In [36]:
val_metrics = latency_metrics_from_log(y_val, final_model.predict(X_val))
test_metrics = latency_metrics_from_log(y_test, final_model.predict(X_test))

pd.DataFrame([val_metrics, test_metrics], index=["val", "test"])

,rmse_log,rmse_ms,rmse_percent,median_relative_error,p90_relative_error,p95_relative_error,median_percent_error,p90_percent_error,p95_percent_error,median_ratio_error,p90_ratio_error,within_10pct,within_25pct,within_50pct,within_2x
val,0.217240,137.833497,29.168820,0.025792,0.235704,0.440950,2.579185,23.570424,44.094959,1.026142,1.270609,0.762696,0.905786,0.958339,0.976726
test,0.210574,122.643537,24.119354,0.024944,0.239314,0.439023,2.494416,23.931415,43.902325,1.025220,1.274954,0.767351,0.904855,0.960015,0.977284


In [19]:
import joblib
from pathlib import Path

In [24]:
bundle = {
    "model": best_model,
    "feature_cols": feature_cols,
    "target_col": "log_average_ms",
}

model_dir = Path("/content/drive/MyDrive/Models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(bundle, model_dir / "3.joblib")

['/content/drive/MyDrive/Models/3.joblib']

# Analysis

In [25]:
test_results = test_df.copy()

test_results["pred_log_latency"] = test_pred
test_results["true_log_latency"] = y_test

test_results["pred_latency_ms"] = np.exp(test_results["pred_log_latency"])
test_results["true_latency_ms"] = np.exp(test_results["true_log_latency"])

test_results["relative_error"] = (
    np.abs(test_results["pred_latency_ms"] - test_results["true_latency_ms"])
    / test_results["true_latency_ms"]
)

In [26]:
def group_latency_metrics(g):
    y_true = g["true_log_latency"].to_numpy()
    y_pred = g["pred_log_latency"].to_numpy()

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    return pd.Series({
        "count": len(g),
        "rmse_log": np.sqrt(mean_squared_error(y_true, y_pred)),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
    })

In [27]:
test_results.groupby("platform").apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6417/3061731483.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby("platform").apply(group_latency_metrics).reset_index()


,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,bluehive,10744.0,0.254159,0.77029,0.877048,0.938198
1,gcloud,10739.0,0.155167,0.76441,0.932675,0.981842


In [28]:
test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6417/2224414100.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()


,machine_type,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,epyc,gcloud,9388.0,0.142962,0.765871,0.940030,0.985407
1,xeon,gcloud,1351.0,0.222173,0.754256,0.881569,0.957069
2,xeon_plat,bluehive,10744.0,0.254159,0.770290,0.877048,0.938198


In [29]:
test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6417/3658681541.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()


,cpu_provider,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,9388.0,0.142962,0.765871,0.940030,0.985407
1,intel,bluehive,10744.0,0.254159,0.770290,0.877048,0.938198
2,intel,gcloud,1351.0,0.222173,0.754256,0.881569,0.957069


In [30]:
test_results.groupby(["cpu_provider", "platform", "num_cores"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,cpu_provider,platform,num_cores,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,1,2636.0,0.092362,0.889226,0.972686,0.993930
1,amd,gcloud,2,4015.0,0.145136,0.742964,0.947447,0.987547
2,amd,gcloud,4,2737.0,0.176032,0.680672,0.897698,0.974059
3,intel,bluehive,1,1347.0,0.208162,0.793615,0.936897,0.963623
4,intel,bluehive,2,2686.0,0.205823,0.848101,0.926657,0.960536
5,intel,bluehive,4,3999.0,0.231187,0.788697,0.888472,0.942486
6,intel,bluehive,8,2712.0,0.337075,0.654499,0.781342,0.897124
7,intel,gcloud,6,1351.0,0.222173,0.754256,0.881569,0.957069
